In [0]:
   spark.range(5).show()

In [0]:
lake_path = "abfss://clean@ukcompanyrisk2026.dfs.core.windows.net/companies/"

df = spark.read.parquet(lake_path)
print(f"Rows: {df.count():,}")
display(df.limit(10))

In [0]:
catalog = spark.sql("SELECT current_catalog()").first()[0]
print("Catalog:", catalog)

for layer in ["bronze", "silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{layer}")
    print(f"Schema ready: {catalog}.{layer}")

In [0]:
from pyspark.sql import functions as F

raw_path = "abfss://raw@ukcompanyrisk2026.dfs.core.windows.net/bulk/"

bronze = (
    spark.read.parquet(raw_path)
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

bronze.write.mode("overwrite").saveAsTable(f"{catalog}.bronze.companies_raw")

print(f"Bronze rows: {spark.table(f'{catalog}.bronze.companies_raw').count():,}")